In [20]:
import numpy as np
from scipy.optimize import linprog

In [21]:
def quantize_state(x, S_n):
    diffs = np.abs(x - S_n)
    return S_n[np.argmin(diffs)]

In [22]:
def quantize_measure(u, l):
    return round(u * l)/ l

In [23]:
def dirac(value):
    mu = np.zeros(n +1)
    idx = np.where(S_n == value)[0][0]
    mu[idx] = 1
    return mu

In [24]:
def phi_n():
    pass

In [25]:
def W2(p, q):
    #note the cost is defined for P(S_n) x P(S_n)

    c = cost_matrix.flatten()
    
    p_len = len(p)
    q_len = len(q)

    if(c.shape[0] != p_len * q_len):
        print("Ensure that the shape of the inputs match the cost matrix/S_n shape")

    A_eq = np.zeros((p_len + q_len, p_len *q_len))

    #we isolate matrix rows
    for i in range(p_len):
        A_eq[i, i * q_len : (i+1) * q_len] = 1.0

    #we isolate matrix cols
    for j in range(q_len):
        A_eq[p_len + j, j::q_len] = 1.0 

    #rows must equal p, cols must equal q
    b_eq = np.concatenate((p, q))

    result = linprog(c, A_eq = A_eq, b_eq = b_eq, bounds = (0, None), method ='highs')
    return result.fun

In [ ]:
#data-label pairs
x = [[1,2],
     [1,2],
     [2,3],
     [3,4],
     [2,3],]
y_tilde = [3.1, 2.9, 4.0, 4.9, 3.9]
# S is the interval [0, 5]

K = len(x)
N = len(x[0])

In [27]:
#1

D_tilde = list(zip(x, y_tilde))

# time horizon = number of layers 
T = 2

# probability measure quantization parameter
# (number of bins = number of quantization levels - 1) (ie l = 1 gives {0, 1})
l = 8

#state space quantization parameter
# (number of bins = number of quantization points - 1) (ie n = 2 gives {0, 1, 2})
n = 5

#action space quantization parameter
m = 5

In [28]:
#2

#create example quantized state space
S_n = []
for i in range(n + 1):
    S_n.append(i)
S_n = np.array(S_n)


def Q_n(x):
    return (quantize_state(x, S_n))
S_n

array([0, 1, 2, 3, 4, 5])

In [29]:
#used for W2
cost_matrix = np.array([[np.linalg.norm(s1 - s2)**2 for s1 in S_n] for s2 in S_n]) 

In [30]:
#3

U_m = {-2, -1, 0, 1, 2}
#for our example, we add the weights

In [31]:
#4

#assume l > 0
P_l = {i/(l) for i in range(0, l+1)}

# for u e P(X_n)
def R_l(u):
    quantized = [quantize_measure(u_k, l) for u_k in u]
    return quantized


In [32]:
#5-7

mu0 = [] #shape = (T, K, N, len(S_n))
y_hat = []
k_unique = 0

#create empirical distributions, remove duplicates, update K to reflect the number of unique labels 
for k in range(K):
    
    candidate_x = np.array([dirac(Q_n(x[k][i])) for i in range(N)])
    candidate_y = dirac(Q_n(y_tilde[k]))

    if k > 0:
        matches = np.all(np.all(np.array(mu0) == candidate_x, axis = 2), axis=1)
    else:
        matches = []

    if np.any(matches):
        idx = np.where(matches == True)[0][0] # we choose the first occurance
        y_hat[idx] += candidate_y
    
    else:
        y_hat.append(candidate_y)
        mu0.append(candidate_x)
        k_unique += 1

K = k_unique

for k in range(K):
    denom = np.sum(y_hat[k])
    y_hat[k] /= denom


In [34]:
y_hat

[array([0., 0., 0., 1., 0., 0.]),
 array([0. , 0. , 0. , 0.5, 0.5, 0. ]),
 array([0., 0., 0., 0., 0., 1.])]